# Análise de Resultados do Método Híbrido Estéreo (Didático)

Este notebook tem como objetivo permitir a visualização e análise dos resultados dos experimentos de estéreo híbrido (Multifoco + Fotométrico) de forma didática e configurável.

## Funcionalidades:
1.  **Carregamento de Configurações**: Lê os parâmetros de experimento diretamente dos arquivos `.yaml`.
2.  **Visualização dos Dados de Entrada**: Exibe as imagens utilizadas no experimento (stack focal ou estéreo fotométrico).
3.  **Visualização dos Resultados**: Apresenta os mapas de altura (profundidade), normais e albedo gerados.
4.  **Análise de Erro**: Se houver gabarito (ground truth), calcula e exibe métricas de erro.

In [1]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2

# Adiciona o diretório src ao path para importar os módulos do projeto
project_root = Path("../../").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.hybrid_stereo_method.infrastructure.io.image_io import (
    read_fni_to_image_array, 
    read_yaml_parameters, 
    read_image,
    find_all_files
)
from src.hybrid_stereo_method.infrastructure.visualization import display_image

## 1. Configuração do Experimento

Defina o caminho para o arquivo de configuração do experimento que deseja analisar. O padrão é `configs/hb_experiment.yaml`.

In [2]:
# Caminho para o arquivo de configuração
config_path = project_root / "configs/hb_experiment.yaml"

# Carrega parâmetros
params = read_yaml_parameters(config_path)

print(f"Experiment Type: {params['experiment']['type']}")
print(f"Input Path: {params['experiment']['paths']['input']}")
print(f"Output Path: {params['experiment']['paths']['output']}")
print(f"Data Folder: {params['experiment']['paths']['data_folder']}")

Experiment Type: hybrid
Input Path: /home/lelis/Documents/Projetos/hybrid-stereo-method/data/raw/hybrid_stereo/
Output Path: /home/lelis/Documents/Projetos/hybrid-stereo-method/data/results/hybrid_stereo/
Data Folder: 2025-03-08-stQ-melon24-amb0.00-glo0.50


## 2. Visualização dos Dados de Entrada

Abaixo visualizamos algumas das imagens de entrada para garantir que o dataset foi carregado corretamente.

In [3]:
input_base_path = Path(params['experiment']['paths']['input'])
data_folder = params['experiment']['paths']['data_folder']
experiment_type = params['experiment']['type']

# Constrói o caminho completo para a pasta de dados
input_data_path = input_base_path / data_folder

print(f"Reading images from: {input_data_path}")

# Lista imagens (filtro simples por extensão png ou jpg)
image_files = sorted([f for f in input_data_path.glob("*.png")])
if not image_files:
    image_files = sorted([f for f in input_data_path.glob("*.jpg")])

# Seleciona algumas imagens para exibir
if len(image_files) > 0:
    # Mostra a primeira, a do meio e a última imagem
    indices_to_show = [0, len(image_files)//2, len(image_files)-1]
    unique_indices = sorted(list(set(indices_to_show)))
    
    plt.figure(figsize=(15, 5))
    for i, idx in enumerate(unique_indices):
        if idx < len(image_files):
            img_path = image_files[idx]
            img = read_image(img_path)
            
            plt.subplot(1, len(unique_indices), i+1)
            if len(img.shape) == 3:
                plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            else:
                plt.imshow(img, cmap='gray')
            plt.title(f"Input {idx}: {img_path.name}")
            plt.axis('off')
    plt.show()
else:
    print("Nenhuma imagem encontrada no diretório de entrada.")

Reading images from: /home/lelis/Documents/Projetos/hybrid-stereo-method/data/raw/hybrid_stereo/2025-03-08-stQ-melon24-amb0.00-glo0.50
Nenhuma imagem encontrada no diretório de entrada.


## 3. Visualização dos Resultados

Aqui carregamos os arquivos `.fni` gerados pelo método. Os arquivos principais são:
*   `height_map.fni`: O mapa de altura recuperado.
*   `normal_map.fni`: O mapa de normais recuperado.
*   `albedo.fni`: O albedo (reflectância) da superfície (se disponível).

In [5]:
output_base_path = Path(params['experiment']['paths']['output'])
result_path = output_base_path / data_folder / "hybrid"

# Se o tipo for apenas multifocus ou photometric, o caminho pode variar levemente
if not result_path.exists():
    # Tenta caminho direto ou subpasta do tipo
    if (output_base_path / data_folder / experiment_type).exists():
        result_path = output_base_path / data_folder / experiment_type
    else:
        result_path = output_base_path / data_folder

print(f"Loading results from: {result_path}")

height_map_path = result_path / "height_map.fni"
normal_map_path = result_path / "normal_map.fni"
albedo_path = result_path / "albedo.fni"
height_iters_dir = result_path / "height-iters"

# Carregar Mapa de Altura
if height_map_path.exists():
    height_map = read_fni_to_image_array(height_map_path)
    print(f"Height Map Loaded. Shape: {height_map.shape}, Min: {height_map.min():.4f}, Max: {height_map.max():.4f}")
else:
    height_map = None
    print(f"Height map not found at {height_map_path}")

# Carregar Mapa de Normais
if normal_map_path.exists():
    normal_map = read_fni_to_image_array(normal_map_path)
    print(f"Normal Map Loaded. Shape: {normal_map.shape}")
else:
    normal_map = None
    print(f"Normal map not found at {normal_map_path}")

# Carregar Albedo
if albedo_path.exists():
    albedo = read_fni_to_image_array(albedo_path)
    print(f"Albedo Loaded. Shape: {albedo.shape}")
else:
    albedo = None
    print("Albedo map not found (this is common for some methods).")

Loading results from: /home/lelis/Documents/Projetos/hybrid-stereo-method/data/results/hybrid_stereo/2025-03-08-stQ-melon24-amb0.00-glo0.50
Height map not found at /home/lelis/Documents/Projetos/hybrid-stereo-method/data/results/hybrid_stereo/2025-03-08-stQ-melon24-amb0.00-glo0.50/height_map.fni
Normal map not found at /home/lelis/Documents/Projetos/hybrid-stereo-method/data/results/hybrid_stereo/2025-03-08-stQ-melon24-amb0.00-glo0.50/normal_map.fni
Albedo map not found (this is common for some methods).


### 3.1 Exibição Gráfica

In [6]:
plt.figure(figsize=(18, 6))

if height_map is not None:
    plt.subplot(1, 3, 1)
    plt.imshow(height_map, cmap='viridis')
    plt.colorbar(label='Height (mm)')
    plt.title("Height Map")
    plt.axis('off')

if normal_map is not None:
    plt.subplot(1, 3, 2)
    # Normal maps usually have values in [-1, 1] or [0, 1]. 
    # Visualization often requires normalizing to [0, 1] range for RGB display.
    # Assuming normals are NxMx3 vector fields
    
    # Normalize for display: (n + 1) / 2 shifts [-1, 1] to [0, 1]
    normal_vis = (normal_map + 1) / 2.0
    # Clip just in case
    normal_vis = np.clip(normal_vis, 0, 1)
    
    plt.imshow(normal_vis)
    plt.title("Normal Map (RGB)")
    plt.axis('off')

if albedo is not None:
    plt.subplot(1, 3, 3)
    plt.imshow(albedo, cmap='gray')
    plt.colorbar(label='Intensity')
    plt.title("Albedo")
    plt.axis('off')

plt.tight_layout()
plt.show()

<Figure size 1800x600 with 0 Axes>

### 3.2 Visualização 3D (Superfície)

Uma visualização em superfície da altura recuperada.

In [7]:
if height_map is not None:
    # Downsample for faster rendering if too large
    h, w = height_map.shape
    skip = 4  # Ajuste para renderizar msis pontos se necessário
    
    X = np.arange(0, w, skip)
    Y = np.arange(0, h, skip)
    X, Y = np.meshgrid(X, Y)
    Z = height_map[::skip, ::skip]

    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X, Y, Z, cmap='viridis', linewidth=0, antialiased=False)
    
    fig.colorbar(surf, shrink=0.5, aspect=5, label='Height')
    ax.set_title("3D Surface Reconstruction")
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Height')
    plt.show()

## 4. Evolução das Iterações (Se disponível)

Se o modo de debug estiver ativado, podemos visualizar como o mapa de altura evoluiu ao longo das iterações.

In [8]:
if height_iters_dir.exists():
    iter_files = sorted([f for f in height_iters_dir.glob("*.fni")])
    
    if iter_files:
        print(f"Found {len(iter_files)} iteration files.")
        
        # Mostrar evolução: Início, Meio, Fim
        indices_iters = [0, len(iter_files)//2, len(iter_files)-1]
        unique_iters = sorted(list(set(indices_iters)))
        
        plt.figure(figsize=(15, 5))
        for i, idx in enumerate(unique_iters):
            f_path = iter_files[idx]
            h_iter = read_fni_to_image_array(f_path)
            
            plt.subplot(1, len(unique_iters), i+1)
            plt.imshow(h_iter, cmap='viridis')
            plt.title(f"Iter {idx}: {f_path.name}")
            plt.axis('off')
        plt.show()
    else:
        print("No iteration files found inside height-iters folder.")
else:
    print("Debug iteration folder 'height-iters' not found.")

Debug iteration folder 'height-iters' not found.
